# ECHO ARABIC — دوس يا طريق | Free Colab Lyric Video

نسخة مجانية بالكامل تعمل على Google Colab، بدون Creative Claw credits.

**النتيجة:** فيديو عمودي 720×1280 / 30fps، مستمر بلا فراغات أو شاشة سوداء، الصوت الأصلي كامل، كلمات عربية متزامنة، وانتقالات Crossfade، مع Logo `ECHO ARABIC`.

شغّل **Runtime → Run all**. يفضّل T4 GPU من **Runtime → Change runtime type**.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!apt-get -qq update
!apt-get -qq install -y ffmpeg fonts-noto-core
!pip -q install faster-whisper mutagen requests

from pathlib import Path
ROOT = Path('/content/drive/MyDrive/ECHO_ARABIC_Dos_Ya_Tareeq')
SRC = ROOT / 'src'
WORK = ROOT / 'work'
OUT = ROOT / 'output'
for p in (SRC, WORK, OUT):
    p.mkdir(parents=True, exist_ok=True)

print('Project folder:', ROOT)


In [ ]:
AUDIO_URL = 'https://cdn.creativeclaw.co/u/2eb76212/audio/f1cf35f1-b782-4160-9dcf-fd18fe26e251.mp3'
VIDEO_URLS = {
  "night1": "https://cdn.creativeclaw.co/u/2eb76212/videos/fa51da72-d5cc-4978-be0f-a533733d6109.mp4",
  "night2": "https://cdn.creativeclaw.co/u/2eb76212/videos/ca11d0c5-8b6a-4563-8772-7dd8501c394e.mp4",
  "night3": "https://cdn.creativeclaw.co/u/2eb76212/videos/24120068-d357-4d5c-90f2-2e6104fdc688.mp4",
  "friends": "https://cdn.creativeclaw.co/u/2eb76212/videos/53bd9e96-7b18-433b-b33a-a1c78f13e2af.mp4",
  "sunrise_city": "https://cdn.creativeclaw.co/u/2eb76212/videos/904668b0-3061-41e8-ba06-784fd7cdb0c0.mp4",
  "sunrise_road": "https://cdn.creativeclaw.co/u/2eb76212/videos/df967382-e534-4509-9772-3b27c25d81be.mp4"
}
LYRICS = '[Intro Hook]\nماشي في الليل\nوالشارع ساكت\nقلبي تقيل\nبس لسه ثابت\n\n[Verse 1]\nعديت ليالي كانت طويلة\nوشوفت وشوش كانت جميلة\nناس قالت هتفضل معايا\nوأول طريق سابت إيديا\n\nأنا مش ناسي\nبس مش واقف\nجرح اللي فات\nعلمني أعارف\n\nكل باب اتقفل في وشي\nخلاني أدور على طريقي\nوكل مرة وقعت فيها\nرجعت أقوى من اللي قبلي\n\n[Pre-Chorus]\nوالليل مهما يطول\nآخره نور\nوالقلب مهما يتعب\nيرجع يدور\n\nأنا لسه ماشي\nأنا لسه حي\nوأول خيط نور\nجايلي من بعيد\n\n[Chorus]\nدوس يا طريق\nأنا جاي لك\nمهما التعب\nأنا مكمل\n\nدوس يا زمان\nمش هتهدني\nأنا اللي وقعت\nوقمت لوحدي\n\nوالصبح جاي\nشايفه قدامي\nواللي راح راح\nوالجاي أيامي\n\nدوس يا طريق\nوسع مكانك\nأنا راجع أقوى\nمن زمانك\n\n[Break]\nعلى مهلي\nبس ما برجعش\nلو قلبي تعب\nبرضه ما بوقعش\n\n[Verse 2]\nقهوة على الرصيف في سكة\nضحكة صاحب تفك الضيقة\nناس بسيطة حواليا\nترجع الروح من تاني فيا\n\nباب محل بيتفتح بدري\nوالنهار بيبان في صدري\nكنت شايف الدنيا آخرها\nطلع آخر الليل هو بدري\n\nأنا مش طالب غير راحتي\nولا طالب حد يعيش مكاني\nأنا عايز أمشي وأنا عارف\nإن بكرة أحسن من زماني\n\n[Build]\nخلي الخطوة تعلى شوية\nخلي النور يدخل في عينيا\nكل اللي فات بقى ورايا\nوأنا قدامك يا دنيا جايلك\n\nواحد\nاتنين\nيلا نكمل\nقلبنا صاحي\nومش هنبطل\n\n[Final Chorus]\nدوس يا طريق\nأنا جاي لك\nمهما التعب\nأنا مكمل\n\nدوس يا زمان\nمش هتهدني\nأنا اللي وقعت\nوقمت لوحدي\n\nوالصبح طلع\nفوق المدينة\nواللي تعبنا\nقوانا فينا\n\nإحنا اللي عشنا\nوإحنا اللي كملنا\nوآخر الطريق\nلقينا نفسنا\n\nدوس يا طريق\nوسع مكانك\nأنا راجع أقوى\nمن زمانك\n\n[Outro]\nوالصبح جاي\nوالصبح جاي\nأنا لسه ماشي\nومش راجع تاني'

W, H, FPS = 720, 1280, 30
CROSSFADE = 0.35
WHISPER_MODEL = 'large-v3-turbo'


In [ ]:
import requests, subprocess, json, os, math

def download(url, path):
    path = Path(path)
    if path.exists() and path.stat().st_size > 100_000:
        print('cached:', path.name)
        return
    print('downloading:', path.name)
    with requests.get(url, stream=True, timeout=180) as r:
        r.raise_for_status()
        with open(path, 'wb') as f:
            for chunk in r.iter_content(1024*1024):
                if chunk:
                    f.write(chunk)

download(AUDIO_URL, SRC/'song.mp3')
for name, url in VIDEO_URLS.items():
    download(url, SRC/f'{name}.mp4')

print('All sources ready.')


In [ ]:
def ffprobe_duration(path):
    cmd = ['ffprobe','-v','error','-show_entries','format=duration','-of','json',str(path)]
    return float(json.loads(subprocess.check_output(cmd))['format']['duration'])

AUDIO_DUR = ffprobe_duration(SRC/'song.mp3')
print(f'Audio duration: {AUDIO_DUR:.3f}s')


In [ ]:
from faster_whisper import WhisperModel
import torch, re, difflib

device = 'cuda' if torch.cuda.is_available() else 'cpu'
compute = 'float16' if device == 'cuda' else 'int8'
print('device:', device, 'compute:', compute)

model = WhisperModel(WHISPER_MODEL, device=device, compute_type=compute)
segments, info = model.transcribe(
    str(SRC/'song.mp3'),
    language='ar',
    beam_size=5,
    word_timestamps=True,
    vad_filter=False,
    condition_on_previous_text=True
)

asr_words = []
for seg in segments:
    for w in (seg.words or []):
        txt = (w.word or '').strip()
        if txt:
            asr_words.append({'text': txt, 'start': float(w.start), 'end': float(w.end)})

print('ASR words:', len(asr_words))
print('Preview:', ' '.join(w['text'] for w in asr_words[:25]))


In [ ]:
AR_DIAC = re.compile(r'[\u0617-\u061A\u064B-\u0652\u0670\u0640]')
PUNCT = re.compile(r'[^\w\u0600-\u06FF]+', re.UNICODE)

def norm_ar(s):
    s = AR_DIAC.sub('', s)
    s = s.replace('أ','ا').replace('إ','ا').replace('آ','ا').replace('ى','ي').replace('ؤ','و').replace('ئ','ي')
    return PUNCT.sub('', s).strip()

lyric_lines = []
for raw in LYRICS.splitlines():
    t = raw.strip()
    if not t or (t.startswith('[') and t.endswith(']')):
        continue
    words = [w for w in t.split() if norm_ar(w)]
    if words:
        lyric_lines.append({'text': t, 'words': words})

exact_tokens = [w for line in lyric_lines for w in line['words']]
A = [norm_ar(x) for x in exact_tokens]
B = [norm_ar(w['text']) for w in asr_words]

sm = difflib.SequenceMatcher(a=A, b=B, autojunk=False)
times = [None] * len(A)

for tag, i1, i2, j1, j2 in sm.get_opcodes():
    if tag == 'equal':
        for k in range(i2-i1):
            times[i1+k] = (asr_words[j1+k]['start'], asr_words[j1+k]['end'])
    elif tag == 'replace':
        used = set()
        for i in range(i1, i2):
            best = None
            for j in range(j1, j2):
                if j in used:
                    continue
                score = difflib.SequenceMatcher(None, A[i], B[j]).ratio()
                if best is None or score > best[0]:
                    best = (score, j)
            if best and best[0] >= 0.55:
                j = best[1]
                used.add(j)
                times[i] = (asr_words[j]['start'], asr_words[j]['end'])

known = [i for i,t in enumerate(times) if t]
if not known:
    raise RuntimeError('Whisper found no usable timing anchors.')

for i,t in enumerate(times):
    if t:
        continue
    left = max([k for k in known if k < i], default=None)
    right = min([k for k in known if k > i], default=None)
    if left is not None and right is not None:
        step = max(0.12, (times[right][0] - times[left][1]) / max(1, right-left))
        st = times[left][1] + step * max(0, i-left-1)
        en = min(st + step*0.85, times[right][0]-0.02)
        times[i] = (st, max(st+0.08, en))
    elif left is not None:
        st = times[left][1] + 0.45 * (i-left)
        times[i] = (st, st+0.38)
    else:
        en = max(0.1, times[right][0] - 0.45*(right-i))
        times[i] = (max(0, en-0.38), en)

prev = 0.0
for i,(st,en) in enumerate(times):
    st = max(prev, st)
    en = max(st+0.08, en)
    en = min(en, AUDIO_DUR)
    times[i] = (st,en)
    prev = st

cursor = 0
for line in lyric_lines:
    n = len(line['words'])
    line['word_times'] = [(*times[cursor+i], line['words'][i]) for i in range(n)]
    line['start'] = max(0.0, line['word_times'][0][0]-0.08)
    line['end'] = min(AUDIO_DUR, line['word_times'][-1][1]+0.18)
    cursor += n

print('Aligned lyric lines:', len(lyric_lines))
for x in lyric_lines[:10]:
    print(f"{x['start']:6.2f}-{x['end']:6.2f}  {x['text']}")


In [ ]:
def ass_time(sec):
    sec = max(0,float(sec))
    h = int(sec//3600); sec -= h*3600
    m = int(sec//60); sec -= m*60
    return f'{h}:{m:02d}:{sec:05.2f}'

def ass_escape(s):
    return s.replace('\\','\\\\').replace('{','\\{').replace('}','\\}')

header = (
    "[Script Info]\n"
    "ScriptType: v4.00+\n"
    "PlayResX: 720\n"
    "PlayResY: 1280\n"
    "WrapStyle: 2\n"
    "ScaledBorderAndShadow: yes\n\n"
    "[V4+ Styles]\n"
    "Format: Name,Fontname,Fontsize,PrimaryColour,SecondaryColour,OutlineColour,BackColour,Bold,Italic,Underline,StrikeOut,ScaleX,ScaleY,Spacing,Angle,BorderStyle,Outline,Shadow,Alignment,MarginL,MarginR,MarginV,Encoding\n"
    "Style: Lyric,Noto Sans Arabic,54,&H00FFFFFF,&H0029C5F6,&H00101010,&H60000000,-1,0,0,0,100,100,0,0,1,4,1,2,55,55,170,1\n"
    "Style: Brand,DejaVu Sans,22,&H00FFFFFF,&H00FFFFFF,&H00101010,&H50000000,-1,0,0,0,100,100,2,0,1,2,0,9,30,30,35,1\n\n"
    "[Events]\n"
    "Format: Layer,Start,End,Style,Name,MarginL,MarginR,MarginV,Effect,Text\n"
)

events = []
events.append(
    f"Dialogue: 3,0:00:00.00,{ass_time(AUDIO_DUR)},Brand,,0,0,0,,"
    "{\\c&H00D7FF&}ECHO{\\c&HFFFFFF&} ARABIC"
)

for line in lyric_lines:
    parts = []
    for st,en,word in line['word_times']:
        cs = max(8, int(round((en-st)*100)))
        parts.append(r'{\kf'+str(cs)+'}'+ass_escape(word))
    text = ' '.join(parts)
    text = r'{\fad(120,180)\fscx96\fscy96\t(0,160,\fscx100\fscy100)}' + text
    events.append(
        f"Dialogue: 2,{ass_time(line['start'])},{ass_time(line['end'])},Lyric,,0,0,0,,{text}"
    )

ass_path = WORK/'lyrics.ass'
ass_path.write_text(header + '\n'.join(events) + '\n', encoding='utf-8-sig')
print('Subtitle file:', ass_path)


In [ ]:
import random
random.seed(7)

src_durs = {name: ffprobe_duration(SRC/f'{name}.mp4') for name in VIDEO_URLS}
print(src_durs)

phase_sources = [
    (0.00,0.43,['night1','night2','night3']),
    (0.43,0.70,['friends','night3','night2']),
    (0.70,1.00,['sunrise_city','sunrise_road','friends'])
]
base_lengths = [5.2,4.1,6.0,3.8,5.6,4.6,6.4,4.0,5.0,4.4]

shots = []
covered = 0.0
idx = 0
while covered < AUDIO_DUR + 1.5:
    progress = min(0.999, covered/AUDIO_DUR)
    choices = next(srcs for a,b,srcs in phase_sources if a <= progress < b)
    name = choices[idx % len(choices)]
    L = base_lengths[idx % len(base_lengths)]
    dur = src_durs[name]
    max_start = max(0.0, dur-L-0.15)
    start = (idx*3.17) % max(0.01,max_start) if max_start > 0.01 else 0.0
    shots.append((name,start,L))
    covered += L - (CROSSFADE if idx > 0 else 0)
    idx += 1

print('shots:', len(shots), 'timeline:', covered)

clip_paths = []
for i,(name,start,L) in enumerate(shots):
    out = WORK/f'clip_{i:03d}.mp4'
    clip_paths.append(out)
    if out.exists() and out.stat().st_size > 100_000:
        continue
    cmd = [
        'ffmpeg','-y','-hide_banner','-loglevel','error',
        '-ss',f'{start:.3f}','-i',str(SRC/f'{name}.mp4'),
        '-t',f'{L:.3f}','-an',
        '-vf',f"scale={W}:{H}:force_original_aspect_ratio=increase,crop={W}:{H},fps={FPS},format=yuv420p",
        '-c:v','libx264','-preset','veryfast','-crf','22',
        str(out)
    ]
    subprocess.run(cmd, check=True)

print('Normalized clips ready at original speed.')


In [ ]:
inputs = []
for p in clip_paths:
    inputs += ['-i', str(p)]

filters = [f'[{i}:v]settb=AVTB[v{i}]' for i in range(len(clip_paths))]
current = 'v0'
timeline_end = shots[0][2]

for i in range(1, len(shots)):
    offset = timeline_end - CROSSFADE
    out = f'x{i}'
    filters.append(
        f'[{current}][v{i}]xfade=transition=fade:duration={CROSSFADE}:offset={offset:.3f}[{out}]'
    )
    current = out
    timeline_end += shots[i][2] - CROSSFADE

montage = WORK/'montage_continuous.mp4'
cmd = [
    'ffmpeg','-y','-hide_banner','-loglevel','error',
    *inputs,
    '-filter_complex',';'.join(filters),
    '-map',f'[{current}]','-an',
    '-c:v','libx264','-preset','veryfast','-crf','21','-pix_fmt','yuv420p',
    str(montage)
]
subprocess.run(cmd, check=True)
print('Continuous montage:', montage)
print('Montage duration:', ffprobe_duration(montage))


In [ ]:
final_path = OUT/'Dos_Ya_Tareeq_ECHO_ARABIC_720x1280.mp4'
ass_filter = str(ass_path).replace('\\','\\\\').replace(':','\\:').replace("'","\\'")

cmd = [
    'ffmpeg','-y','-hide_banner','-loglevel','error',
    '-i',str(montage),
    '-i',str(SRC/'song.mp3'),
    '-vf',f"subtitles='{ass_filter}'",
    '-map','0:v:0','-map','1:a:0',
    '-t',f'{AUDIO_DUR:.3f}',
    '-c:v','libx264','-preset','medium','-crf','20','-pix_fmt','yuv420p',
    '-c:a','aac','-b:a','192k',
    '-movflags','+faststart',
    str(final_path)
]
subprocess.run(cmd, check=True)
print('FINAL:', final_path)


In [ ]:
VIDEO_DUR = ffprobe_duration(final_path)
delta = VIDEO_DUR - AUDIO_DUR
print(f'Audio: {AUDIO_DUR:.3f}s | Video: {VIDEO_DUR:.3f}s | delta: {delta:+.3f}s')

if VIDEO_DUR < AUDIO_DUR - 0.08:
    raise RuntimeError('FAIL: output video ends before the song.')

probe = subprocess.check_output([
    'ffprobe','-v','error','-select_streams','v:0',
    '-show_entries','frame=best_effort_timestamp_time',
    '-of','json',str(final_path)
])
frames = json.loads(probe).get('frames', [])
ts = [float(x['best_effort_timestamp_time']) for x in frames if 'best_effort_timestamp_time' in x]
gaps = [b-a for a,b in zip(ts,ts[1:])]
max_gap = max(gaps) if gaps else 0.0
expected = 1/FPS

print(f'Frames: {len(ts)} | expected step ~{expected:.4f}s | max timestamp step {max_gap:.4f}s')
if max_gap > expected * 2.2:
    raise RuntimeError(f'FAIL: abnormal video timestamp gap detected: {max_gap:.3f}s')

print('✅ QA PASSED: no timeline gaps and video covers the full song.')


In [ ]:
from IPython.display import Video, display
display(Video(str(final_path), embed=False, width=360))
print('Saved permanently in Google Drive:')
print(final_path)


In [ ]:
from google.colab import files
# Optional: uncomment only if you want to download to the phone.
# files.download(str(final_path))
